# FIN-001 — 02 · Experiments & Results

Model Gate evidence: the run comparison, the frozen candidate, and the protected-test result with error analysis. Full write-up in `reports/model_gate.md`.

This notebook READS the records produced by `python -m src.experiments` and `python -m src.finalize` (re-run those first if the files are missing).

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # repo root importable
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
pd.set_option('display.max_columns', 20)

## 1. Experiment record (selection on validation)

In [2]:
rec = pd.read_csv('../reports/experiment_record.csv')
rec[['run','kind','valid_macro_f1','valid_accuracy','fit_predict_seconds']]

,run,kind,valid_macro_f1,valid_accuracy,fit_predict_seconds
0,tfidf_char_linsvm,tfidf,0.9971,0.9968,4.6
1,tfidf_word_logreg,tfidf,0.9960,0.9955,1.2
2,tfidf_word_complementnb,tfidf,0.9915,0.9905,0.2
3,embed_minilm_logreg,embed,0.9468,0.9395,46.8
4,baseline_majority,baseline,0.0094,0.0867,0.0


**Interpretation.** Every TF-IDF model beats the majority baseline (macro-F1 0.009) by ~100x. The character-n-gram SVM is marginally best, but it has no calibrated probabilities — which the brief's confidence-based fallback needs — so the deployed candidate is `tfidf_word_logreg`. The MiniLM sentence-embedding run is *worse* (0.947) and far heavier: on short structured merchant strings, sparse bag-of-words wins. Selection uses validation only; the test set is untouched here.

## 2. Protected-test result (scored once in finalize.py)

In [3]:
import json
m = json.load(open('../artifacts/metrics.json'))
print('final run:', m['final_run'], '| version', m['model_version'])
print('TEST:', m['test'])
print('unseen-merchant slice:', m['test_unseen_merchants'], 'n=', m['n_test_unseen_merchant'])

final run: tfidf_word_logreg | version 1.0.0
TEST: {'accuracy': 0.9955, 'macro_f1': 0.9961, 'weighted_f1': 0.9955, 'n': 6854, 'top3_accuracy': 0.999}
unseen-merchant slice: {'accuracy': 0.993, 'macro_f1': 0.9784, 'weighted_f1': 0.993, 'n': 716} n= 716


**Honest reading.** Test macro-F1 0.9961 is strong but optimistic — the data is templated around ~500 merchants. The unseen-merchant slice (0.9784) is the realistic generalization number and the gap is the expected real-world degradation.

## 3. Where it errs

In [4]:
per_class = pd.read_csv('../reports/results/per_class_test.csv')
per_class.head(5)  # weakest categories first

,category,precision,recall,f1,support
0,Healthcare,0.984252,0.984252,0.984252,381.0
1,Shopping,0.994863,0.976471,0.985581,595.0
2,Groceries,0.983302,0.988806,0.986047,536.0
3,Subscription,0.989101,0.994521,0.991803,365.0
4,Transfer,0.993088,1.000000,0.996532,431.0


In [5]:
conf = pd.read_csv('../reports/results/top_confusions_test.csv')
conf.head(10)

,true,predicted,count
0,Shopping,Healthcare,5
1,Shopping,Groceries,4
2,Healthcare,Groceries,3
3,Subscription,Groceries,2
4,Shopping,Subscription,2
5,Shopping,Personal Care,2
6,Healthcare,Shopping,2
7,Restaurants,Subscription,1
8,Shopping,Transfer,1
9,Restaurants,Transfer,1


**Conclusion.** Confusions cluster on genuinely overlapping merchants (e.g. Groceries vs Shopping). The product mitigation is the confidence + OOV fallback in `src/inference.py`, which routes low-confidence or unrecognized transactions to human review instead of auto-applying a guess. See `reports/error_analysis/ERROR_ANALYSIS.md`.